# 08 · Backpropagation

**Learning objectives**
1. Run a forward pass through a $2\to2\to1$ network and record every intermediate value.
2. Derive each gradient with the chain rule, step by step.
3. Verify every gradient with a finite-difference check, and see that one update lowers the loss.

Modules: `src/mlp.py`, `tests/test_mlp.py`.

## 1. Setup

Network $2\to2\to1$, sigmoid hidden and output units, binary cross-entropy loss. Input $\mathbf{x}=(1.0, 0.5)$, target $y=1$.

$$z^{(1)}=\mathbf{x}W^{(1)}+\mathbf{b}^{(1)},\quad h=\1(z^{(1)}),\quad z^{(2)}=hW^{(2)}+b^{(2)},\quad \hat y=\1(z^{(2)}),\quad L=-\log\hat y$$

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

from mlp import TwoLayerNet

net = TwoLayerNet(
    W1=np.array([[0.1, 0.4], [0.3, -0.2]]),
    b1=np.array([0.0, 0.1]),
    W2=np.array([[0.5], [-0.3]]),
    b2=np.array([0.2]),
)
x = np.array([1.0, 0.5])
y = 1.0
cache = net.forward(x)
print("z1 =", cache["z1"], " h =", cache["h"].round(4))
print("z2 =", cache["z2"].round(4), " y_hat =", cache["y_hat"].round(4))
print("loss =", round(net.loss(x, y), 4))

z1 = [0.25 0.4 ]  h = [0.5622 0.5987]
z2 = [0.3015]  y_hat = [0.5748]
loss = 0.5537


## 2. Backward pass, step by step

1. $\1 L/\1 z^{(2)} = \hat y - y$ (BCE with sigmoid, notebook 06).
2. $\1 L/\1 W^{(2)}_j = h_j\,\1 L/\1 z^{(2)}$, and $\1 L/\1 b^{(2)} = \1 L/\1 z^{(2)}$.
3. $\1 L/\1 h_j = W^{(2)}_j\,\1 L/\1 z^{(2)}$ (one output, one path).
4. $\1 L/\1 z^{(1)}_j = \1 L/\1 h_j \cdot h_j(1-h_j)$.
5. $\1 L/\1 W^{(1)}_{ij} = x_i\,\1 L/\1 z^{(1)}_j$, and $\1 L/\1 b^{(1)}_j = \1 L/\1 z^{(1)}_j$.

The module implements these lines. The next cell prints the gradients with the intermediate values.

In [2]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

grads, loss = net.backward(x, y)
dz2 = cache["y_hat"] - y
print("dL/dz2 =", dz2.round(4))
print("dL/dW2 =", grads["W2"].ravel().round(4), "  dL/db2 =", grads["b2"].round(4))
dh = net.W2[:, 0] * dz2
print("dL/dh  =", dh.round(4))
dz1 = dh * cache["h"] * (1 - cache["h"])
print("dL/dz1 =", dz1.round(4))
print("dL/dW1 =\n", grads["W1"].round(4))
print("dL/db1 =", grads["b1"].round(4))

dL/dz2 = [-0.4252]
dL/dW2 = [-0.239  -0.2546]   dL/db2 = [-0.4252]
dL/dh  = [-0.2126  0.1276]
dL/dz1 = [-0.0523  0.0306]
dL/dW1 =
 [[-0.0523  0.0306]
 [-0.0262  0.0153]]
dL/db1 = [-0.0523  0.0306]


## 3. Gradient check

A central finite difference $\frac{L(w+\varepsilon)-L(w-\varepsilon)}{2\varepsilon}$ should match the analytic gradient for every parameter.

In [3]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

eps = 1e-6
rows = []
for name, arr in net.params().items():
    it = np.nditer(arr, flags=["multi_index"])
    for _ in it:
        idx = it.multi_index
        orig = arr[idx]
        arr[idx] = orig + eps; lp = net.loss(x, y)
        arr[idx] = orig - eps; lm = net.loss(x, y)
        arr[idx] = orig
        numeric = (lp - lm) / (2 * eps)
        rows.append((f"{name}{idx}", numeric, grads[name][idx]))
worst = max(abs(n - a) for _, n, a in rows)
for label, n, a in rows:
    print(f"{label:12s} numeric={n:+.8f}  analytic={a:+.8f}")
print("max |numeric - analytic| =", f"{worst:.1e}")

W1(0, 0)     numeric=-0.05232752  analytic=-0.05232752
W1(0, 1)     numeric=+0.03064732  analytic=+0.03064732
W1(1, 0)     numeric=-0.02616376  analytic=-0.02616376
W1(1, 1)     numeric=+0.01532366  analytic=+0.01532366
b1(0,)       numeric=-0.05232752  analytic=-0.05232752
b1(1,)       numeric=+0.03064732  analytic=+0.03064732
W2(0, 0)     numeric=-0.23903478  analytic=-0.23903478
W2(1, 0)     numeric=-0.25455915  analytic=-0.25455915
b2(0,)       numeric=-0.42519525  analytic=-0.42519525
max |numeric - analytic| = 1.0e-10


## 4. One gradient step, then a loss curve

Update every parameter with learning rate $0.5$. The loss should fall.

In [4]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

from copy import deepcopy
trained = deepcopy(net)
losses = [trained.loss(x, y)]
for _ in range(20):
    g, _ = trained.backward(x, y)
    for name, arr in trained.params().items():
        arr -= 0.5 * g[name]
    losses.append(trained.loss(x, y))
print("loss before:", round(losses[0], 4), " after 1 step:", round(losses[1], 4), " after 20 steps:", round(losses[-1], 4))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(losses, marker="o", ms=3); ax.set_xlabel("gradient step"); ax.set_ylabel("loss")
ax.set_title("Loss during gradient descent (lr = 0.5)"); ax.grid(True)
plt.show()

loss before: 0.5537  after 1 step: 0.414  after 20 steps: 0.051


## 5. Why depth makes gradients smaller

Each sigmoid layer multiplies the gradient by a derivative at most $0.25$. The product over 10 layers is at most $0.25^{10}$.

In [5]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

print("upper bound on product of 10 sigmoid derivatives:", f"{0.25**10:.3e}")

upper bound on product of 10 sigmoid derivatives: 9.537e-07


## 6. Interpretation

- **Mathematical:** the chain rule gives every gradient as a product of local factors, and the forward values are reused.
- **Verified:** the analytic gradients match finite differences; a step lowers the loss.
- **Engineering:** the same pattern scales to any depth, but each layer multiplies the gradient, which causes vanishing or exploding gradients (module 8 in the roadmap).

## 7. Exercises

1. Predict whether the loss decreases if the learning rate is $5.0$. Run it and check.
2. Set $y=0$ and recompute $\\1 L/\\1 z^{(2)}$ by hand.
3. Show that if $h_1(1-h_1)$ were replaced by $1$, the finite-difference check would fail. Explain why.